In [2]:
import os
import re
from collections import defaultdict

import torch
import numpy as np
import pandas as pd
from transformers import pipeline

from models import Bleurt, Mpnet, ModernBERT, ModernBERT_v2, ModelInput

torch.set_float32_matmul_precision("high")
os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [3]:
mb_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_authentic_multirc"
mb_w_reference_answer_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_authentic_multirc_with_reference_with_label"

pipes = {
    "Mpnet_pipe": Mpnet(),
    "Bleurt_pipe": Bleurt(),
    "ModernBERT_pipe": ModernBERT(mb_path),
    "ModernBERT_with_reference": ModernBERT_v2(mb_w_reference_answer_path),
}

Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda


In [4]:
responses = pd.read_csv("../../bin/2025-09-09T21-23_export.csv", index_col=0)
content = pd.read_csv("../../data/cri_questions2.csv")
df = responses.merge(content, how="inner", on=["chunk_slug", "page_slug"]).rename(
    columns={
        "text": "candidate",
        "answer": "reference",
        "chunk_text": "text",
    }
)
df

,candidate,score,condition,user_id,page_slug,chunk_slug,is_passed,chapter,created_at,volume_title,volume_slug,page_title,chunk_header,text,question,reference
0,khgkhghj,2.171278,stairs,fhdqbtxevjpxteuwt4iu3u7e2m,the-u-s-constitution,Supreme-Law-of-the-Land-4459,True,0,2025-09-08T22:14:41.667Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,Supreme Law of the Land,The sixth section of the Constitution says the...,What does the sixth section of the Constitutio...,The Constitution is the supreme law of the lan...
1,the supreme law of the land,2.000000,stairs,wf5jicpb5uon42yzcgjpptpzyu,the-u-s-constitution,Supreme-Law-of-the-Land-4459,True,0,2025-09-04T22:07:47.238Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,Supreme Law of the Land,The sixth section of the Constitution says the...,What does the sixth section of the Constitutio...,The Constitution is the supreme law of the lan...
2,I don't know,1.812920,stairs,fhdqbtxevjpxteuwt4iu3u7e2m,the-u-s-constitution,State-Governments-4457,False,0,2025-09-08T22:13:06.217Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...
3,The role of the executive branch is enforce fe...,2.778327,stairs,ci7hvzzqif5lhd4becodqngqye,the-u-s-constitution,State-Governments-4457,True,0,2025-09-08T03:42:18.481Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...
4,a,2.000000,stairs,ezc4c5yzimt7izgd3ugov2nyma,the-u-s-constitution,State-Governments-4457,True,0,2025-09-03T15:29:57.897Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...
5,n,0.000000,stairs,ezc4c5yzimt7izgd3ugov2nyma,the-u-s-constitution,State-Governments-4457,True,0,2025-09-03T15:29:51.531Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...
6,1776,2.631567,stairs,i6iirxwoh4e3bj4pkzlxyeobse,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-08T08:06:55.561Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.
7,dasd,2.403645,stairs,6aadzk7jqpgmu4fpa44ajcxfze,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-05T17:51:29.110Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.
8,1787,2.000000,stairs,sievio4tkp2lqjadtpracywgtq,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-04T22:02:12.351Z,"One Nation, One People: The USCIS Civics Test ...",one-nation-one-people-the-uscis-civics-test-te...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nT

In [5]:
def score(df, pipes, label_key="labels"):
    all_preds = defaultdict(list)

    for name, pipe in pipes.items():
        for ex in df.to_dict("records"):
            model_input = ModelInput.from_dict(ex)
            pred = pipe(model_input)
            all_preds[name].append(pred)

    return all_preds


all_preds = score(df, pipes)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [6]:
for model_name, preds in all_preds.items():
    df[model_name] = preds

df["ensemble"] = df["Mpnet_pipe"] + df["Bleurt_pipe"]
df

,candidate,score,condition,user_id,page_slug,chunk_slug,is_passed,chapter,created_at,volume_title,...,page_title,chunk_header,text,question,reference,Mpnet_pipe,Bleurt_pipe,ModernBERT_pipe,ModernBERT_with_reference,ensemble
0,khgkhghj,2.171278,stairs,fhdqbtxevjpxteuwt4iu3u7e2m,the-u-s-constitution,Supreme-Law-of-the-Land-4459,True,0,2025-09-08T22:14:41.667Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,Supreme Law of the Land,The sixth section of the Constitution says the...,What does the sixth section of the Constitutio...,The Constitution is the supreme law of the lan...,0,0,2.170975,1.770094,0
1,the supreme law of the land,2.000000,stairs,wf5jicpb5uon42yzcgjpptpzyu,the-u-s-constitution,Supreme-Law-of-the-Land-4459,True,0,2025-09-04T22:07:47.238Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,Supreme Law of the Land,The sixth section of the Constitution says the...,What does the sixth section of the Constitutio...,The Constitution is the supreme law of the lan...,1,1,2.628478,3.115588,2
2,I don't know,1.812920,stairs,fhdqbtxevjpxteuwt4iu3u7e2m,the-u-s-constitution,State-Governments-4457,False,0,2025-09-08T22:13:06.217Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...,0,0,1.813844,1.492057,0
3,The role of the executive branch is enforce fe...,2.778327,stairs,ci7hvzzqif5lhd4becodqngqye,the-u-s-constitution,State-Governments-4457,True,0,2025-09-08T03:42:18.481Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...,0,1,2.777066,2.599865,1
4,a,2.000000,stairs,ezc4c5yzimt7izgd3ugov2nyma,the-u-s-constitution,State-Governments-4457,True,0,2025-09-03T15:29:57.897Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...,1,1,1.470363,1.832988,2
5,n,0.000000,stairs,ezc4c5yzimt7izgd3ugov2nyma,the-u-s-constitution,State-Governments-4457,True,0,2025-09-03T15:29:51.531Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,State Governments,The fourth section of the Constitution explain...,What is the role of the executive branch in a ...,The person in charge of the executive branch f...,0,0,1.295934,1.149224,0
6,1776,2.631567,stairs,i6iirxwoh4e3bj4pkzlxyeobse,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-08T08:06:55.561Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.,0,0,2.631322,1.983289,0
7,dasd,2.403645,stairs,6aadzk7jqpgmu4fpa44ajcxfze,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-05T17:51:29.110Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.,0,0,2.402570,1.395974,0
8,1787,2.000000,stairs,sievio4tkp2lqjadtpracywgtq,the-u-s-constitution,The-US-Constitution-4445,True,0,2025-09-04T22:02:12.351Z,"One Nation, One People: The USCIS Civics Test ...",...,Chapter 1: The U.S. Constitution,The U.S. Constitution,"In this chapter, you will learn about:\n\nThe ...",When was the U.S. Constitution written?,The U.S. Constitution was written in 1787.,1,1,2.987468,2.697045,2
9,in the year alkjdf,0.000000,stairs,siev

In [9]:
df.to_csv("../../data/us-civics-trial-data-v2.csv")